# 🔋 Notebook 1 — Gerador de Dataset: Setor de Energia
### Workshop Databricks para Times de Negócio

Este notebook gera um dataset realístico de uma **distribuidora de energia elétrica**, com 3 tabelas:

| Tabela | Descrição |
|---|---|
| **clientes** | Cadastro de clientes residenciais, comerciais e industriais |
| **consumo** | Histórico mensal de consumo (kWh), demanda e valores faturados |
| **instalacoes** | Equipamentos elétricos instalados por cliente |

Os dados são salvos como **tabelas Delta no Unity Catalog** e como **arquivos CSV em um Volume**.

### 📋 Descrição das Colunas — Tabela CLIENTES

 Tabela      | Coluna                | Descrição                                                                 |
-------------|----------------------|---------------------------------------------------------------------------|
 clientes    | cliente_id           | Identificador único do cliente                                            |
 clientes    | nome                 | Nome do cliente                                                           |
 clientes    | tipo_cliente         | Tipo de cliente (Residencial, Comercial, Industrial)                      |
 clientes    | cidade               | Cidade do cliente                                                         |
 clientes    | estado               | Estado do cliente                                                         |
 clientes    | tensao               | Nível de tensão da ligação                                                |
 clientes    | data_cadastro        | Data de cadastro do cliente                                               |
 clientes    | ativo                | Status de atividade do cliente                                            |

### 📋 Descrição das Colunas — Tabela CONSUMO

 Tabela      | Coluna                | Descrição                                                                 |
-------------|----------------------|---------------------------------------------------------------------------|
 consumo     | consumo_id           | Identificador único do registro de consumo                                |
 consumo     | cliente_id           | Identificador do cliente                                                  |
 consumo     | data_leitura         | Data da leitura                                                           |
 consumo     | mes                  | Mês da leitura                                                            |
 consumo     | ano                  | Ano da leitura                                                            |
 consumo     | kwh_consumido        | Quantidade de energia consumida (kWh)                                     |
 consumo     | demanda_kw           | Demanda de energia (kW)                                                   |
 consumo     | valor_rs             | Valor da fatura em reais                                                  |
 consumo     | status_fatura        | Status da fatura (Paga, Pendente, Vencida)                                |

### 📋 Descrição das Colunas — Tabela INSTALACOES

 Tabela      | Coluna                | Descrição                                                                 |
-------------|----------------------|---------------------------------------------------------------------------|
 instalacoes | instalacao_id         | Identificador único da instalação                                         |
 instalacoes | cliente_id            | Identificador do cliente                                                  |
 instalacoes | tipo_equipamento      | Tipo de equipamento instalado                                             |
 instalacoes | fabricante            | Fabricante do equipamento                                                 |
 instalacoes | capacidade_kw         | Capacidade do equipamento (kW)                                            |
 instalacoes | data_instalacao       | Data de instalação                                                        |
 instalacoes | ultima_manutencao     | Data da última manutenção                                                 |
 instalacoes | status                | Status do equipamento                                                     |
 instalacoes | vida_util_anos        | Vida útil estimada em anos                                                |

## ⚙️ Passo 1 — Configurar Parâmetros (Widgets)

Preencha os widgets abaixo com as informações do seu ambiente:
- **nome_catalogo**: Catálogo do Unity Catalog onde as tabelas serão criadas
- **nome_schema**: Schema (banco de dados) onde as tabelas serão criadas
- **seu_prefixo**: Prefixo único para evitar conflitos entre participantes (ex: `carol_`)
- **caminho_volume**: Caminho do Volume onde os CSVs serão salvos

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text("nome_catalogo","","Catálogo")
dbutils.widgets.text("nome_schema","","Schema")
dbutils.widgets.text("seu_prefixo","","Seu Prefixo")
dbutils.widgets.text("caminho_volume", "", "Caminho do Volume")

In [0]:
nome_catalogo  = dbutils.widgets.get("nome_catalogo")
nome_schema    = dbutils.widgets.get("nome_schema")
seu_prefixo    = dbutils.widgets.get("seu_prefixo")
caminho_volume = dbutils.widgets.get("caminho_volume")

print(f"Catálogo : {nome_catalogo}")
print(f"Schema   : {nome_schema}")
print(f"Prefixo  : {seu_prefixo}")
print(f"Volume   : {caminho_volume}")

## 📦 Passo 2 — Importações

In [0]:
import random
import datetime
import pandas as pd
import numpy as np
from pyspark.sql.types import *
import os


random.seed(42)
np.random.seed(42)

## 🗄️ Passo 3 — Criar Schema e Volume no Unity Catalog

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {nome_catalogo}.{nome_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {nome_catalogo}.{nome_schema}.arquivos")

print(f"✅ Schema criado : {nome_catalogo}.{nome_schema}")
print(f"✅ Volume criado : {nome_catalogo}.{nome_schema}.arquivos")

## 🏗️ Passo 4 — Gerar os Dados

### Tabela 1: `clientes`
Cadastro de 200 clientes de diferentes tipos (Residencial, Comercial, Industrial), distribuídos em cidades brasileiras.

In [0]:
CIDADES = [
    ("São Paulo", "SP"),      ("Rio de Janeiro", "RJ"), ("Belo Horizonte", "MG"),
    ("Salvador", "BA"),        ("Fortaleza", "CE"),      ("Curitiba", "PR"),
    ("Manaus", "AM"),          ("Recife", "PE"),          ("Porto Alegre", "RS"),
    ("Goiânia", "GO"),         ("Campinas", "SP"),        ("São Luís", "MA"),
    ("Maceió", "AL"),          ("Natal", "RN"),           ("Teresina", "PI"),
    ("Campo Grande", "MS"),    ("João Pessoa", "PB"),     ("Aracaju", "SE"),
    ("Cuiabá", "MT"),          ("Porto Velho", "RO"),
]

NOMES_PRIMEIRO = ["Ana", "Carlos", "Maria", "João", "Pedro", "Luciana", "Roberto",
                   "Fernanda", "Marcelo", "Patricia", "Rafael", "Juliana", "Bruno",
                   "Camila", "Diego", "Vanessa", "Felipe", "Larissa", "Gustavo", "Aline"]
NOMES_SEGUNDO  = ["Silva", "Santos", "Oliveira", "Souza", "Costa", "Ferreira",
                   "Rodrigues", "Almeida", "Nascimento", "Lima", "Araújo", "Carvalho",
                   "Melo", "Barbosa", "Ribeiro", "Pereira", "Lopes", "Gomes", "Martins", "Rocha"]
EMPRESAS_COM   = ["Supermercado", "Padaria", "Farmácia", "Restaurante", "Academia",
                   "Hotel", "Clínica", "Loja de Roupas", "Posto de Gasolina", "Escola"]
EMPRESAS_IND   = ["Metalúrgica", "Fábrica de Plásticos", "Indústria Química",
                   "Frigorífico", "Cimenteira", "Mineradora", "Refinaria", "Siderúrgica"]

NUM_CLIENTES = 200
TIPOS        = ["Residencial", "Comercial", "Industrial"]
TENSOES      = {"Residencial": "Baixa Tensão", "Comercial": "Média Tensão", "Industrial": "Alta Tensão"}

clientes = []
for i in range(1, NUM_CLIENTES + 1):
    tipo   = random.choices(TIPOS, weights=[55, 30, 15])[0]
    cidade, estado = random.choice(CIDADES)

    if tipo == "Residencial":
        nome = f"{random.choice(NOMES_PRIMEIRO)} {random.choice(NOMES_SEGUNDO)}"
    elif tipo == "Comercial":
        nome = f"{random.choice(EMPRESAS_COM)} {random.choice(NOMES_SEGUNDO)}"
    else:
        nome = f"{random.choice(EMPRESAS_IND)} {random.choice(NOMES_SEGUNDO)} Ltda"

    data_cadastro = (datetime.date(2018, 1, 1) + datetime.timedelta(days=random.randint(0, 365 * 6))).strftime("%Y-%m-%d")

    clientes.append({
        "cliente_id":    i,
        "nome":          nome,
        "tipo_cliente":  tipo,
        "cidade":        cidade,
        "estado":        estado,
        "tensao":        TENSOES[tipo],
        "data_cadastro": data_cadastro,
        "ativo":         random.choices([True, False], weights=[90, 10])[0],
    })

df_clientes = pd.DataFrame(clientes)
print(f"Clientes gerados: {len(df_clientes)}")
display(df_clientes.head(5))

### Tabela 2: `consumo`
Histórico mensal de consumo por cliente: kWh consumido, demanda em kW, valor faturado em R$ e status da fatura.
Inclui sazonalidade (verão = maior consumo) e tarifas diferenciadas por nível de tensão.

In [0]:
CONSUMO_BASE = {
    "Residencial": (150,   500),
    "Comercial":   (800,   6_000),
    "Industrial":  (10_000, 80_000),
}
TARIFAS = {
    "Baixa Tensão": 0.78,
    "Média Tensão": 0.52,
    "Alta Tensão":  0.34,
}
SAZONALIDADE = {1: 1.30, 2: 1.25, 3: 1.20, 4: 1.00, 5: 0.95, 6: 0.85,
                7: 0.85, 8: 0.90, 9: 0.95, 10: 1.00, 11: 1.10, 12: 1.25}

consumos    = []
consumo_id  = 1
DATA_INICIO = datetime.date(2024, 1, 1)
DATA_FIM    = datetime.date(2025, 12, 31)

for cliente in clientes:
    num_meses   = random.randint(12, 24)
    inicio_hist = DATA_INICIO - datetime.timedelta(days=30 * (num_meses - 1))

    kwh_min, kwh_max = CONSUMO_BASE[cliente["tipo_cliente"]]
    tarifa           = TARIFAS[cliente["tensao"]]

    for m in range(num_meses):
        data_leitura = inicio_hist + datetime.timedelta(days=30 * m)
        if data_leitura > DATA_FIM:
            break

        sazon      = SAZONALIDADE[data_leitura.month]
        kwh        = round(random.uniform(kwh_min, kwh_max) * sazon, 2)
        valor_rs   = round(kwh * tarifa * random.uniform(0.97, 1.03), 2)
        demanda_kw = round(kwh / (24 * 30) * random.uniform(0.85, 1.15), 2)

        consumos.append({
            "consumo_id":    consumo_id,
            "cliente_id":    cliente["cliente_id"],
            "data_leitura":  data_leitura.strftime("%Y-%m-%d"),
            "mes":           data_leitura.month,
            "ano":           data_leitura.year,
            "kwh_consumido": kwh,
            "demanda_kw":    demanda_kw,
            "valor_rs":      valor_rs,
            "status_fatura": random.choices(
                ["Paga", "Pendente", "Vencida"], weights=[75, 15, 10]
            )[0],
        })
        consumo_id += 1

df_consumo = pd.DataFrame(consumos).head(500)
print(f"Registros de consumo gerados: {len(df_consumo)}")
display(df_consumo.head(5))

### Tabela 3: `instalacoes`
Equipamentos elétricos instalados por cliente: tipo, fabricante, capacidade, data de instalação, última manutenção e status atual.

In [0]:
TIPOS_EQUIP  = ["Medidor Digital", "Transformador", "Quadro de Distribuição",
                 "No-break Industrial", "Gerador Diesel", "Painéis Solares",
                 "Banco de Capacitores", "Disjuntor Geral", "Subestação", "UPS"]
FABRICANTES  = ["Schneider Electric", "ABB", "Siemens", "WEG", "Eaton",
                 "GE", "Philips", "Delta", "Emerson", "Hitachi"]
STATUS_LIST  = ["Ativo", "Em Manutenção", "Desativado", "Substituído"]

instalacoes   = []
instalacao_id = 1

for cliente in clientes:
    if cliente["tipo_cliente"] == "Residencial":
        num_inst = 1
    elif cliente["tipo_cliente"] == "Comercial":
        num_inst = random.randint(1, 3)
    else:
        num_inst = random.randint(2, 5)

    for _ in range(num_inst):
        data_cadastro    = datetime.date.fromisoformat(cliente["data_cadastro"])
        data_instalacao  = data_cadastro + datetime.timedelta(days=random.randint(0, 30))
        ultima_manut     = data_instalacao + datetime.timedelta(days=random.randint(180, 365 * 5))
        if ultima_manut > datetime.date.today():
            ultima_manut = datetime.date.today() - datetime.timedelta(days=random.randint(1, 180))

        cap_max = 10 if cliente["tipo_cliente"] == "Residencial" else (
            500 if cliente["tipo_cliente"] == "Comercial" else 5000
        )

        instalacoes.append({
            "instalacao_id":     instalacao_id,
            "cliente_id":        cliente["cliente_id"],
            "tipo_equipamento":  random.choice(TIPOS_EQUIP),
            "fabricante":        random.choice(FABRICANTES),
            "capacidade_kw":     round(random.uniform(0.5, cap_max), 1),
            "data_instalacao":   data_instalacao.strftime("%Y-%m-%d"),
            "ultima_manutencao": ultima_manut.strftime("%Y-%m-%d"),
            "status":            random.choices(STATUS_LIST, weights=[70, 15, 8, 7])[0],
            "vida_util_anos":    random.randint(5, 25),
        })
        instalacao_id += 1

df_instalacoes = pd.DataFrame(instalacoes).head(500)
print(f"Instalações geradas: {len(df_instalacoes)}")
display(df_instalacoes.head(5))

## 💾 Passo 5 — Salvar Tabelas no Unity Catalog

In [0]:
tabelas = {
    "clientes":    df_clientes,
    "consumo":     df_consumo,
    "instalacoes": df_instalacoes,
}

for nome_tabela, df_pandas in tabelas.items():
    nome_completo = f"{nome_catalogo}.{nome_schema}.{seu_prefixo}{nome_tabela}"
    df_spark = spark.createDataFrame(df_pandas)
    (
        df_spark.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_completo)
    )
    print(f"✅ Tabela salva: {nome_completo}  ({df_pandas.shape[0]} linhas, {df_pandas.shape[1]} colunas)")

## 📁 Passo 6 — Salvar CSVs no Volume

In [0]:
import shutil

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = '/Workspace' + os.path.dirname(notebook_path)

for nome_tabela, df_pandas in tabelas.items():
    caminho_csv = f"/{notebook_dir}/{seu_prefixo}_{nome_tabela}.csv"
    df_pandas.to_csv(caminho_csv, index=False, encoding="utf-8")

# os.makedirs(os.path.dirname(caminho_volume.rstrip('/')), exist_ok=True)
# os.makedirs(caminho_volume, exist_ok=True)

for nome_tabela in tabelas.keys():
    origem = f"/{notebook_dir}/{seu_prefixo}_{nome_tabela}.csv"
    destino = f"{caminho_volume}{seu_prefixo}_{nome_tabela}.csv"
    shutil.copy(origem, destino)
    print(f"✅ Arquivo copiado para o volume: {destino}")

## ✅ Passo 7 — Resumo Final

In [0]:
print("=" * 65)
print("  RESUMO — DATASET DE ENERGIA GERADO COM SUCESSO")
print("=" * 65)

for nome_tabela, df_pandas in tabelas.items():
    nome_completo = f"{nome_catalogo}.{nome_schema}.{seu_prefixo}{nome_tabela}"
    print(f"\n📊 {nome_tabela.upper()}")
    print(f"   Delta Table : {nome_completo}")
    print(f"   CSV         : {caminho_volume}{seu_prefixo}_{nome_tabela}.csv")
    print(f"   Linhas      : {df_pandas.shape[0]}")
    print(f"   Colunas     : {', '.join(df_pandas.columns.tolist())}")

print("\n" + "=" * 65)
print("🎉 Tudo pronto! O dataset está disponível no Unity Catalog.")
print("=" * 65)